# MLPAutoencoder Test Notebook

This notebook tests the new `MLPAutoencoder` class, which implements a nonlinear encoder
for investigating manifold representations in toy models of superposition.

**Architecture:**
- Encoder: `x ∈ ℝⁿ → σ(V · σ(Wx + b₁) + b₂) → h ∈ ℝᵐ`
- Decoder: `h ∈ ℝᵐ → W_dec · h + b_dec → x̂ ∈ ℝⁿ` (linear)

Key features:
- Smooth nonlinearity (GELU/tanh/SiLU) to enable curved feature manifolds
- Linear decoder to isolate nonlinearity to the encoder side
- Compatible with `ToyModel` and `ModelGrid`

In [ ]:
import numpy as np
import torch
from torch import Generator

from occhio import ToyModel, ModelGrid, MLPAutoencoder
from occhio.autoencoder import TiedLinearRelu
from occhio.distributions.sparse import SparseUniform
from occhio.model_grid import Axis
from occhio.visualization.geometry import plot_geometry

## 1. Basic Instantiation and Forward Pass

In [ ]:
# Test basic instantiation
ae = MLPAutoencoder(n_features=20, n_hidden=5)
print(f"MLPAutoencoder:")
print(f"  n_features: {ae.n_features}")
print(f"  n_hidden: {ae.n_hidden}")
print(f"  encoder_hidden_dim: {ae.encoder_hidden_dim}")
print(f"  activation: {ae._activation_name}")
print(f"  total parameters: {sum(p.numel() for p in ae.parameters())}")

In [ ]:
# Test forward pass
x = torch.randn(32, 20)
x_hat, z = ae(x)
print(f"Forward pass shapes:")
print(f"  Input: {x.shape}")
print(f"  Latent: {z.shape}")
print(f"  Output: {x_hat.shape}")

## 2. Different Activation Functions

In [ ]:
# Test different activations
for activation in ["gelu", "tanh", "silu"]:
    ae = MLPAutoencoder(n_features=10, n_hidden=3, activation=activation)
    x = torch.randn(16, 10)
    x_hat, z = ae(x)
    print(f"{activation}: latent mean={z.mean().item():.4f}, std={z.std().item():.4f}")

## 3. Custom Encoder Hidden Dimension

In [ ]:
# Test custom encoder hidden dim
ae_small = MLPAutoencoder(n_features=20, n_hidden=5, encoder_hidden_dim=8)
ae_large = MLPAutoencoder(n_features=20, n_hidden=5, encoder_hidden_dim=64)

print(f"Small encoder hidden dim: {ae_small.encoder_hidden_dim}")
print(f"  Parameters: {sum(p.numel() for p in ae_small.parameters())}")
print(f"Large encoder hidden dim: {ae_large.encoder_hidden_dim}")
print(f"  Parameters: {sum(p.numel() for p in ae_large.parameters())}")

## 4. Integration with ToyModel

In [ ]:
N_FEATURES = 50
N_HIDDEN = 10

generator = Generator().manual_seed(42)

dist = SparseUniform(
    n_features=N_FEATURES,
    p_active=0.1,
    generator=generator,
)

ae = MLPAutoencoder(
    n_features=N_FEATURES,
    n_hidden=N_HIDDEN,
    activation="gelu",
)

tm = ToyModel(distribution=dist, ae=ae)
print(f"ToyModel created with MLPAutoencoder")
print(f"  Distribution: {type(dist).__name__}")
print(f"  Autoencoder: {type(ae).__name__}")

In [ ]:
# Train the model (fit returns tuple: (losses, hook_returns))
losses, _ = tm.fit(n_epochs=2000, batch_size=256, track_losses=True, verbose=True)
print(f"\nFinal loss: {losses[-1]:.6f}")

## 5. Integration with ModelGrid

In [ ]:
N_FEATURES = 100
N_HIDDEN = 20
FEATURE_IMPORTANCE_DECAY = 0.99


def create_mlp_model(params):
    generator = Generator().manual_seed(42)
    return ToyModel(
        distribution=SparseUniform(
            n_features=N_FEATURES,
            p_active=params["Feature Probability"],
            generator=generator,
        ),
        ae=MLPAutoencoder(
            n_features=N_FEATURES,
            n_hidden=N_HIDDEN,
        ),
        importances=torch.tensor(
            [FEATURE_IMPORTANCE_DECAY**i for i in range(N_FEATURES)]
        ),
    )


mlp_grid = ModelGrid(
    create_mlp_model,
    axes=[Axis(label="Feature Probability", values=10 ** np.linspace(0, -2, 5))],
)
print(f"ModelGrid shape: {mlp_grid.shape}")
print(
    f"Feature probabilities: {[f'{v:.3f}' for v in mlp_grid.axes[0].values.tolist()]}"
)

In [ ]:
# Train the grid
mlp_grid.fit(n_epochs=5000, batch_size=256, verbose=True)

In [ ]:
# Visualize geometry (may need adjustment for nonlinear encoders)
try:
    fig = plot_geometry(mlp_grid)
    fig.show()
except Exception as e:
    print(f"Visualization note: {e}")
    print("This may require geometry analysis adjustments for nonlinear encoders.")

## 6. Comparison: MLPAutoencoder vs TiedLinearRelu

In [ ]:
N_FEATURES = 100
N_HIDDEN = 20
N_EPOCHS = 3000
P_ACTIVE = 0.1


# Create comparable models
def create_linear_model():
    generator = Generator().manual_seed(42)
    return ToyModel(
        distribution=SparseUniform(
            n_features=N_FEATURES,
            p_active=P_ACTIVE,
            generator=generator,
        ),
        ae=TiedLinearRelu(n_features=N_FEATURES, n_hidden=N_HIDDEN),
    )


def create_mlp_model_single():
    generator = Generator().manual_seed(42)
    return ToyModel(
        distribution=SparseUniform(
            n_features=N_FEATURES,
            p_active=P_ACTIVE,
            generator=generator,
        ),
        ae=MLPAutoencoder(n_features=N_FEATURES, n_hidden=N_HIDDEN),
    )

In [ ]:
# Train both models
linear_model = create_linear_model()
mlp_model = create_mlp_model_single()

print("Training TiedLinearRelu...")
linear_losses, _ = linear_model.fit(
    n_epochs=N_EPOCHS, batch_size=256, track_losses=True
)
print(f"Final loss: {linear_losses[-1]:.6f}")

print("\nTraining MLPAutoencoder...")
mlp_losses, _ = mlp_model.fit(n_epochs=N_EPOCHS, batch_size=256, track_losses=True)
print(f"Final loss: {mlp_losses[-1]:.6f}")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2, subplot_titles=["Training Loss", "Final Loss Comparison"]
)

# Loss curves
fig.add_trace(
    go.Scatter(y=linear_losses, name="TiedLinearRelu", line=dict(color="blue")),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(y=mlp_losses, name="MLPAutoencoder", line=dict(color="red")),
    row=1,
    col=1,
)

# Final loss comparison
fig.add_trace(
    go.Bar(
        x=["TiedLinearRelu", "MLPAutoencoder"],
        y=[linear_losses[-1], mlp_losses[-1]],
        marker_color=["blue", "red"],
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_layout(height=400, title_text="MLPAutoencoder vs TiedLinearRelu")
fig.update_xaxes(title_text="Epoch", row=1, col=1)
fig.update_yaxes(title_text="Loss", row=1, col=1)
fig.update_yaxes(title_text="Final Loss", row=1, col=2)
fig.show()

## 7. Weight Resampling Test

In [ ]:
# Test resample_weights
ae = MLPAutoencoder(n_features=10, n_hidden=3)
initial_w1 = ae.enc_W1.clone()

ae.resample_weights()
resampled_w1 = ae.enc_W1.clone()

print(f"Weights changed after resample: {not torch.allclose(initial_w1, resampled_w1)}")

## 8. Device Handling

In [ ]:
# Test device handling
ae_cpu = MLPAutoencoder(n_features=10, n_hidden=3, device="cpu")
print(f"CPU device: {ae_cpu.device}")

# Test MPS if available
if torch.backends.mps.is_available():
    ae_mps = MLPAutoencoder(n_features=10, n_hidden=3, device="mps")
    print(f"MPS device: {ae_mps.device}")

    # Test forward on MPS
    x = torch.randn(16, 10, device="mps")
    x_hat, z = ae_mps(x)
    print(f"MPS forward pass successful, output device: {x_hat.device}")

## Summary

The `MLPAutoencoder` class is fully functional and compatible with:
- `ToyModel` for single-model training
- `ModelGrid` for parallel parameter sweeps
- Multiple activation functions (GELU, tanh, SiLU)
- Custom encoder hidden dimensions
- Weight resampling
- Device handling (CPU, MPS)

Next steps for manifold analysis (Phase 2 of the plan):
- Implement Jacobian computation for feature directions
- Compute angular variance to measure nonlinearity
- Compare linear vs nonlinear models for evidence of manifold structure